In [ ]:
# Part (a)
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import Image, display

def load_series(path):
    values = []
    with open(path) as f:
        for token in f.read().split():
            try:
                values.append(float(token))
            except ValueError:
                continue
    return values

def build_regression_matrices(series, order):
    X, y = [], []
    for t in range(order, len(series)):
        row = [1.0]
        for j in range(1, order + 1):
            row.append(series[t - j])
        X.append(row)
        y.append(series[t])
    return X, y

def solve_linear_system(A, b):
    n = len(A)
    aug = [A[i][:] + [b[i]] for i in range(n)]
    for col in range(n):
        pivot_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        aug[col], aug[pivot_row] = aug[pivot_row], aug[col]
        pivot = aug[col][col]
        if abs(pivot) < 1e-12:
            raise ValueError('Singular matrix encountered while solving normal equations.')
        for j in range(col, n + 1):
            aug[col][j] /= pivot
        for r in range(n):
            if r == col:
                continue
            factor = aug[r][col]
            if factor == 0:
                continue
            for j in range(col, n + 1):
                aug[r][j] -= factor * aug[col][j]
    return [aug[i][n] for i in range(n)]

def fit_ar(series, order):
    X, y = build_regression_matrices(series, order)
    p = len(X[0])
    XtX = [[0.0] * p for _ in range(p)]
    XtY = [0.0] * p
    for row, target in zip(X, y):
        for j in range(p):
            XtY[j] += row[j] * target
            for k in range(p):
                XtX[j][k] += row[j] * row[k]
    beta = solve_linear_system(XtX, XtY)
    return beta

def compute_residuals(series, beta, order):
    residuals = [None] * order
    for t in range(order, len(series)):
        pred = beta[0]
        for j in range(1, order + 1):
            pred += beta[j] * series[t - j]
        residuals.append(series[t] - pred)
    return residuals

def autocorrelation(data, lag):
    m = len(data)
    mean = sum(data) / m
    denom = sum((x - mean) ** 2 for x in data)
    num = sum((data[i] - mean) * (data[i - lag] - mean) for i in range(lag, m))
    return num / denom

hare_counts = load_series('hare.dat')
hare_series = [math.sqrt(value) for value in hare_counts]
order = 3
beta = fit_ar(hare_series, order)
residuals = compute_residuals(hare_series, beta, order)
filtered_residuals = [r for r in residuals if r is not None]
max_lag = 20
acf_values = [1.0] + [autocorrelation(filtered_residuals, lag) for lag in range(1, max_lag + 1)]

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
lags = range(1, max_lag + 1)
markerline, stemlines, baseline = ax.stem(lags, acf_values[1:], basefmt=' ')
plt.setp(markerline, marker='o', markersize=5)
plt.setp(stemlines, linewidth=1.2)
ax.axhline(0, color='black', linewidth=1)
conf = 1.96 / math.sqrt(len(filtered_residuals))
ax.axhline(conf, color='red', linestyle='--', linewidth=1)
ax.axhline(-conf, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('Sample ACF of AR(3) Residuals')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylim(min(-1, min(acf_values[1:]) - 0.1), max(1, max(acf_values[1:]) + 0.1))
plt.tight_layout()
acf_path = Path('hare_residuals_acf.jpg')
fig.savefig(acf_path, format='jpg', dpi=300)
plt.close(fig)
display(Image(filename=str(acf_path)))


In [ ]:
# Part (b)
K = 9
N = len(filtered_residuals)
ljung_box_Q = N * (N + 2) * sum((acf_values[k] ** 2) / (N - k) for k in range(1, K + 1))
print(f'Ljung-Box Q-statistic (K=9): {ljung_box_Q:.4f}')
print(f'Degrees of freedom: {K - order}')


In [ ]:
# Part (c)
from statistics import NormalDist

nonzero_residuals = [r for r in filtered_residuals if r != 0]
signs = [1 if r > 0 else -1 for r in nonzero_residuals]
runs = 1
for i in range(1, len(signs)):
    if signs[i] != signs[i - 1]:
        runs += 1
n_pos = sum(1 for s in signs if s == 1)
n_neg = sum(1 for s in signs if s == -1)
expected_runs = 1 + 2 * n_pos * n_neg / (n_pos + n_neg)
variance_runs = (2 * n_pos * n_neg * (2 * n_pos * n_neg - n_pos - n_neg)) / (((n_pos + n_neg) ** 2) * (n_pos + n_neg - 1))
z_runs = (runs - expected_runs) / math.sqrt(variance_runs)
p_runs = 2 * (1 - NormalDist().cdf(abs(z_runs)))
print(f'Runs count: {runs}')
print(f'Expected runs: {expected_runs:.4f}')
print(f'Z-statistic: {z_runs:.4f}')
print(f'Two-sided p-value: {p_runs:.4f}')


In [ ]:
# Part (d)
from statistics import NormalDist
from pathlib import Path

ndist = NormalDist()
sorted_residuals = sorted(filtered_residuals)
N = len(sorted_residuals)
qq_points = []
for i, value in enumerate(sorted_residuals, start=1):
    prob = (i - 0.375) / (N + 0.25)
    theor = ndist.inv_cdf(prob)
    qq_points.append((theor, value))

sum_x = sum(pt[0] for pt in qq_points)
sum_y = sum(pt[1] for pt in qq_points)
sum_xx = sum(pt[0] ** 2 for pt in qq_points)
sum_xy = sum(pt[0] * pt[1] for pt in qq_points)
slope = (N * sum_xy - sum_x * sum_y) / (N * sum_xx - sum_x ** 2)
intercept = (sum_y - slope * sum_x) / N

fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
theor_values = [pt[0] for pt in qq_points]
sample_values = [pt[1] for pt in qq_points]
ax.scatter(theor_values, sample_values, s=25, edgecolor='black', facecolor='royalblue', alpha=0.8)
line_x = [min(theor_values), max(theor_values)]
line_y = [slope * x + intercept for x in line_x]
ax.plot(line_x, line_y, color='firebrick', linewidth=1.5, label='Reference line')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title('QQ Plot of AR(3) Residuals')
ax.legend(loc='upper left')
ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
plt.tight_layout()
qq_path = Path('hare_residuals_qq.jpg')
fig.savefig(qq_path, format='jpg', dpi=300)
plt.close(fig)
display(Image(filename=str(qq_path)))


In [ ]:
# Part (e)
import math
import random
from statistics import NormalDist

ndist = NormalDist()
N = len(filtered_residuals)
sorted_residuals = sorted(filtered_residuals)
coefficients = [ndist.inv_cdf((i - 0.375) / (N + 0.25)) for i in range(1, N + 1)]
norm_factor = math.sqrt(sum(c * c for c in coefficients))
a_weights = [c / norm_factor for c in coefficients]
mean_residual = sum(filtered_residuals) / N
denom = sum((r - mean_residual) ** 2 for r in filtered_residuals)
numer = sum(a * x for a, x in zip(a_weights, sorted_residuals)) ** 2
W_statistic = numer / denom

random.seed(2024)
iterations = 5000
count = 0
for _ in range(iterations):
    sample = sorted(random.gauss(0, 1) for _ in range(N))
    mean_sample = sum(sample) / N
    denom_sample = sum((s - mean_sample) ** 2 for s in sample)
    numer_sample = sum(a * s for a, s in zip(a_weights, sample)) ** 2
    if denom_sample == 0:
        continue
    W_sample = numer_sample / denom_sample
    if W_sample <= W_statistic:
        count += 1
p_value = count / iterations
print(f'Shapiro-Wilk W statistic (Monte Carlo approximation): {W_statistic:.4f}')
print(f'Approximate p-value: {p_value:.4f}')
